In [0]:
%pip install databricks-feature-engineering

In [0]:
# OHLCV standardisation + point-in-time-safe technical indicators.
# Output feature table is registered with the Feature Engineering client (primary key: symbol + trade_ts),
# same pattern as the fraud reference (fe.create_table).
import sys
import os
sys.path.insert(0, os.getcwd())

from databricks.feature_engineering import FeatureEngineeringClient
from utils import load_project, table, widget_or
from features_lib import standardize_ohlcv, build_feature_frame

cfg = load_project()
d = cfg["data"]

dbutils.widgets.text("catalog", "stock_demo")
dbutils.widgets.text("bronze_schema", "bronze")
for k in ["date_col", "symbol_col", "open_col", "high_col", "low_col", "close_col", "volume_col"]:
    dbutils.widgets.text(k, "")      # empty -> conf/project.yml

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
mapping = {k: widget_or(dbutils.widgets.get(k), d[k]) for k in
           ["date_col", "symbol_col", "open_col", "high_col", "low_col", "close_col", "volume_col"]}
print("Column mapping:", mapping)

for s in cfg["schemas"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{s}")

bronze_name = f"{catalog}.{bronze_schema}.{cfg['tables']['bronze'].split('.')[-1]}"
silver_name = table(cfg, "silver", catalog)
feature_name = table(cfg, "features", catalog)

src = spark.table(bronze_name)
clean = standardize_ohlcv(src, mapping)
clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_name)

feat = build_feature_frame(spark.table(silver_name))

In [0]:
fe = FeatureEngineeringClient()
# fe.write_table() only supports mode="merge", so a full rebuild = drop the old table and recreate it.
# (The table is fully derived from silver, so nothing is lost.)
if spark.catalog.tableExists(feature_name):
    spark.sql(f"DROP TABLE {feature_name}")

fe.create_table(
    name=feature_name,
    primary_keys=["symbol", "trade_ts"],
    df=feat,
    description="Technical-indicator features + next-session close label for stock price regression",
)

print("Silver rows:", spark.table(silver_name).count(), "| Feature rows:", spark.table(feature_name).count())
display(spark.table(feature_name).limit(20))